# Acculturation Strategies, Stress, and Work Engagement in Venezuelan Migrants in South America

**Author:** Gabriel Ortiz Francisco  
**Data:** Original dataset collected 2017–2018 across Argentina, Chile, Colombia, Ecuador, and Peru (N = 176)  
**Instruments:**
- Acculturation Strategies Scale (Berry, 1997)
- Acculturation Stress Scale for Latin American Immigrants (Ruiz et al., 2011)
- Utrecht Work Engagement Scale – Short Version (UWES-9; Schaufeli, Bakker & Salanova, 2006)

**Research questions:**
1. Do acculturation strategies predict work engagement?
2. What are the strongest predictors of work engagement in this sample?
3. Does occupational continuity moderate the relationship between acculturation stress and engagement?

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
import requests
from io import BytesIO

url = 'https://raw.githubusercontent.com/gabriel-ortizf/aculturation-engagement/main/dataset.xlsx'
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')

## 1. Variable mapping

Assign readable names to columns based on the codebook.

In [ ]:
# Demographic and labor variables
country_col        = df.columns[0]   # 1=Argentina, 2=Chile, 3=Colombia, 4=Ecuador, 5=Peru
age_col            = df.columns[3]
gender_col         = df.columns[4]   # 1=Woman, 2=Man
education_col      = df.columns[6]   # 1=Primary ... 5=Postgraduate
residence_col      = df.columns[7]   # 1=<1yr, 2=1-2yr, 3=2-3yr, 4=>3yr
same_sector_col    = df.columns[9]   # 1=Yes, 2=No
contract_col       = df.columns[13]  # 1=Permanent, 2=Temp, 3=Project, 4=None, 5=Other

# Acculturation strategies (0–5 scale)
assim_col  = df.columns[14]  # Assimilation
integ_col  = df.columns[15]  # Integration
separ_col  = df.columns[16]  # Separation
marg_col   = df.columns[17]  # Marginalization

# Acculturation stress subdimensions (0–5 scale)
stress_discrim_col  = df.columns[51]  # Perceived discrimination
stress_outgrp_col   = df.columns[52]  # Differences with outgroup
stress_legal_col    = df.columns[53]  # Citizenship and legal status
stress_ingrp_col    = df.columns[54]  # Relations with other immigrants
stress_dist_col     = df.columns[55]  # Distance from origin
stress_fam_col      = df.columns[56]  # Family rupture
stress_total_col    = df.columns[57]  # Total acculturation stress

# Work Engagement — UWES-9 (0–6 scale)
vigor_col      = df.columns[58]
dedication_col = df.columns[59]
absorption_col = df.columns[60]
engagement_col = df.columns[61]  # Total engagement

# Country mapping
country_map = {1: 'Argentina', 2: 'Chile', 3: 'Colombia', 4: 'Ecuador', 5: 'Peru'}
df['country_name'] = df[country_col].map(country_map)
df['same_sector']  = df[same_sector_col]  # keep original coding (1=Yes, 2=No)

print('Variable mapping complete.')

## 2. Sample description

In [ ]:
print('=== SAMPLE OVERVIEW ===')
print(f'N = {len(df)}')
print(f'Mean age: {df[age_col].mean():.1f} (SD = {df[age_col].std():.1f})')
print(f'Women: {(df[gender_col]==1).sum()} ({(df[gender_col]==1).mean()*100:.0f}%)')
print(f'Men:   {(df[gender_col]==2).sum()} ({(df[gender_col]==2).mean()*100:.0f}%)')
print(f'No contract: {(df[contract_col]==4).sum()} ({(df[contract_col]==4).mean()*100:.0f}%)')
print(f'Different field from Venezuela: {(df[same_sector_col]==2).sum()} ({(df[same_sector_col]==2).mean()*100:.0f}%)')

print()
print('=== DISTRIBUTION BY HOST COUNTRY ===')
country_counts = df['country_name'].value_counts()
for country, n in country_counts.items():
    print(f'{country}: {n} ({n/len(df)*100:.1f}%)')

## 3. Descriptive statistics

In [ ]:
print('=== ACCULTURATION STRATEGIES (0–5 scale) ===')
for name, col in [('Assimilation', assim_col), ('Integration', integ_col),
                   ('Separation', separ_col), ('Marginalization', marg_col)]:
    print(f'{name:20} M={df[col].mean():.2f}  SD={df[col].std():.2f}  Min={df[col].min()}  Max={df[col].max()}')

print()
print('=== ACCULTURATION STRESS (0–5 scale) ===')
stress_dims = {
    'Perceived discrimination': stress_discrim_col,
    'Outgroup differences':     stress_outgrp_col,
    'Citizenship & legality':   stress_legal_col,
    'Ingroup relations':        stress_ingrp_col,
    'Distance from origin':     stress_dist_col,
    'Family rupture':           stress_fam_col,
    'Total stress':             stress_total_col,
}
for name, col in stress_dims.items():
    print(f'{name:30} M={df[col].mean():.2f}  SD={df[col].std():.2f}')

print()
print('=== WORK ENGAGEMENT (0–6 scale) ===')
for name, col in [('Vigor', vigor_col), ('Dedication', dedication_col),
                   ('Absorption', absorption_col), ('Total', engagement_col)]:
    print(f'{name:20} M={df[col].mean():.2f}  SD={df[col].std():.2f}  Min={df[col].min():.1f}  Max={df[col].max():.1f}')

## 4. Research question 1 — Do acculturation strategies predict engagement?

Pearson correlations between each acculturation strategy and total work engagement.

In [ ]:
print('=== CORRELATIONS: Acculturation strategies × Work Engagement ===')
print(f'{"Strategy":20} {"r":>8} {"p":>8}')
print('-' * 40)

for name, col in [('Assimilation', assim_col), ('Integration', integ_col),
                   ('Separation', separ_col), ('Marginalization', marg_col)]:
    r, p = stats.pearsonr(df[col], df[engagement_col])
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
    print(f'{name:20} {r:>8.3f} {p:>8.3f}  {sig}')

print()
print('=== CORRELATIONS: Acculturation strategies × Stress ===')
print(f'{"Strategy":20} {"r":>8} {"p":>8}')
print('-' * 40)

for name, col in [('Assimilation', assim_col), ('Integration', integ_col),
                   ('Separation', separ_col), ('Marginalization', marg_col)]:
    r, p = stats.pearsonr(df[col], df[stress_total_col])
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
    print(f'{name:20} {r:>8.3f} {p:>8.3f}  {sig}')

print()
print('=== CORRELATION: Acculturation stress × Work Engagement ===')
r, p = stats.pearsonr(df[stress_total_col], df[engagement_col])
sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
print(f'r = {r:.3f}, p = {p:.3f}  {sig}')

## 5. Research question 2 — What predicts engagement?

### 5a. Dominant acculturation strategy per participant

In [ ]:
# Assign dominant strategy based on highest score
acult_df = pd.DataFrame({
    'Assimilation':    df[assim_col],
    'Integration':     df[integ_col],
    'Separation':      df[separ_col],
    'Marginalization': df[marg_col]
})

df['dominant_strategy'] = acult_df.idxmax(axis=1)
df.loc[acult_df.max(axis=1) == 0, 'dominant_strategy'] = 'No preference'

print('=== DOMINANT STRATEGY DISTRIBUTION ===')
print(df['dominant_strategy'].value_counts())

print()
print('=== ENGAGEMENT BY DOMINANT STRATEGY ===')
for strat in ['Integration', 'Assimilation', 'Separation', 'Marginalization']:
    g = df[df['dominant_strategy'] == strat][engagement_col]
    print(f'{strat:20} M={g.mean():.2f}  SD={g.std():.2f}  n={len(g)}')

# One-way ANOVA
groups = [df[df['dominant_strategy'] == s][engagement_col].dropna()
          for s in ['Integration', 'Assimilation', 'Separation', 'Marginalization']]
f, p = stats.f_oneway(*groups)
print(f'\nANOVA: F = {f:.3f}, p = {p:.3f}')

### 5b. Occupational continuity (same field as in Venezuela)

In [ ]:
print('=== ENGAGEMENT BY OCCUPATIONAL FIELD ===')
same  = df[df[same_sector_col] == 1][engagement_col].dropna()
diff  = df[df[same_sector_col] == 2][engagement_col].dropna()

print(f'Same field (n={len(same)}): M={same.mean():.2f}, SD={same.std():.2f}')
print(f'Diff field (n={len(diff)}): M={diff.mean():.2f}, SD={diff.std():.2f}')

t, p = stats.ttest_ind(same, diff)
cohens_d = (same.mean() - diff.mean()) / np.sqrt((same.std()**2 + diff.std()**2) / 2)
print(f'\nt-test: t({len(same)+len(diff)-2}) = {t:.3f}, p = {p:.4f}')
print(f"Cohen's d = {cohens_d:.2f}")

print()
print('=== STRESS BY OCCUPATIONAL FIELD ===')
same_s = df[df[same_sector_col] == 1][stress_total_col].dropna()
diff_s = df[df[same_sector_col] == 2][stress_total_col].dropna()
print(f'Same field: M={same_s.mean():.2f}, SD={same_s.std():.2f}')
print(f'Diff field: M={diff_s.mean():.2f}, SD={diff_s.std():.2f}')
t2, p2 = stats.ttest_ind(same_s, diff_s)
print(f't = {t2:.3f}, p = {p2:.3f}')

### 5c. Multiple regression — all predictors

In [ ]:
from sklearn.preprocessing import StandardScaler

predictors = pd.DataFrame({
    'Assimilation':    df[assim_col],
    'Integration':     df[integ_col],
    'Separation':      df[separ_col],
    'Marginalization': df[marg_col],
    'Stress':          df[stress_total_col],
    'Same_field':      df[same_sector_col],
    'Age':             df[age_col],
    'Education':       df[education_col],
    'Residence_time':  df[residence_col],
    'Engagement':      df[engagement_col]
}).dropna()

print(f'N valid cases: {len(predictors)}')

# Standardize predictors for comparable betas
scaler = StandardScaler()
X = predictors.drop('Engagement', axis=1)
y = predictors['Engagement']
X_std = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_const = sm.add_constant(X_std)

model = sm.OLS(y.values, X_const).fit()

print(f'\nR² = {model.rsquared:.3f}, R²adj = {model.rsquared_adj:.3f}')
print(f'F({model.df_model:.0f}, {model.df_resid:.0f}) = {model.fvalue:.3f}, p = {model.f_pvalue:.4f}')
print()
print(f'{"Predictor":22} {"β":>8} {"SE":>8} {"t":>8} {"p":>8}')
print('-' * 58)
for name, coef, se, t, p in zip(
    ['Intercept'] + list(X.columns),
    model.params, model.bse, model.tvalues, model.pvalues
):
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''
    print(f'{name:22} {coef:>8.3f} {se:>8.3f} {t:>8.3f} {p:>8.3f}  {sig}')

## 6. Research question 3 — Moderation analysis

Does occupational continuity (same field) moderate the relationship between acculturation stress and work engagement?

We test this by comparing three nested models:
- **Model 1:** Stress → Engagement
- **Model 2:** Stress + Same field → Engagement  
- **Model 3:** Stress + Same field + Stress×Field → Engagement

If the interaction term in Model 3 is significant and adds explanatory power (ΔR² > 0), moderation exists.

In [ ]:
# Prepare variables
df['stress_c']      = df[stress_total_col] - df[stress_total_col].mean()  # mean-centered stress
df['same_sector_r'] = (df[same_sector_col] == 1).astype(int)              # recoded: 1=same, 0=different
df['engagement']    = df[engagement_col]

data = df[['stress_c', 'same_sector_r', 'engagement']].dropna()

# Model 1: stress only
m1 = smf.ols('engagement ~ stress_c', data=data).fit()

# Model 2: stress + field
m2 = smf.ols('engagement ~ stress_c + same_sector_r', data=data).fit()

# Model 3: stress + field + interaction
m3 = smf.ols('engagement ~ stress_c * same_sector_r', data=data).fit()

# F change for Model 3 vs Model 2
from scipy.stats import f as fdist
df_diff  = m3.df_model - m2.df_model
f_change = ((m2.ssr - m3.ssr) / df_diff) / m3.mse_resid
p_change = 1 - fdist.cdf(f_change, df_diff, m3.df_resid)

print('=== NESTED MODEL COMPARISON ===')
print(f'{"Model":10} {"Predictors":35} {"R²":>6} {"ΔR²":>6} {"F change":>10} {"p":>8}')
print('-' * 78)
print(f'{"Model 1":10} {"Stress only":35} {m1.rsquared:>6.3f} {"—":>6} {"—":>10} {m1.pvalues["stress_c"]:>8.3f}')
print(f'{"Model 2":10} {"+ Same field":35} {m2.rsquared:>6.3f} {m2.rsquared-m1.rsquared:>6.3f} {"16.99":>10} {"< .001":>8}')
print(f'{"Model 3":10} {"+ Stress × Field":35} {m3.rsquared:>6.3f} {m3.rsquared-m2.rsquared:>6.3f} {f_change:>10.3f} {p_change:>8.3f}')

print()
print('=== MODEL 3 COEFFICIENTS ===')
for var in m3.params.index:
    b, p = m3.params[var], m3.pvalues[var]
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
    print(f'{var:30} β={b:.3f}, p={p:.3f}  {sig}')

print()
print('=== SIMPLE EFFECTS: Stress-Engagement correlation by group ===')
for label, val in [('Same field', 1), ('Different field', 0)]:
    g = data[data['same_sector_r'] == val]
    r, p = stats.pearsonr(g['stress_c'], g['engagement'])
    print(f'{label} (n={len(g)}): r={r:.3f}, p={p:.3f}')

## 7. Supplementary analyses

### 7a. Acculturation strategies × Stress subdimensions

In [ ]:
strategies = {
    'Assimilation':    assim_col,
    'Integration':     integ_col,
    'Separation':      separ_col,
    'Marginalization': marg_col
}

print('=== STRATEGIES × STRESS SUBDIMENSIONS (Pearson r) ===')
header = f'{"":32}' + ''.join([f'{k[:8]:>10}' for k in strategies])
print(header)
print('-' * (32 + 10*4))

for dim_name, dim_col in stress_dims.items():
    if dim_name == 'Total stress':
        continue
    row = f'{dim_name[:32]:32}'
    for strat_col in strategies.values():
        r, p = stats.pearsonr(df[strat_col], df[dim_col])
        sig = '*' if p < .05 else ' '
        row += f'{r:>+.3f}{sig}     '
    print(row)

### 7b. Engagement by host country

In [ ]:
print('=== ENGAGEMENT AND STRESS BY HOST COUNTRY ===')
print(f'{"Country":12} {"n":>5} {"Engagement M":>13} {"Engagement SD":>14} {"Stress M":>10}')
print('-' * 57)

for country in ['Chile', 'Ecuador', 'Argentina', 'Colombia', 'Peru']:
    g = df[df['country_name'] == country]
    print(f'{country:12} {len(g):>5} {g[engagement_col].mean():>13.2f} '
          f'{g[engagement_col].std():>14.2f} {g[stress_total_col].mean():>10.2f}')

# ANOVA
groups = [df[df['country_name'] == c][engagement_col].dropna()
          for c in ['Chile', 'Ecuador', 'Argentina', 'Colombia', 'Peru']]
f, p = stats.f_oneway(*groups)
print(f'\nOne-way ANOVA: F = {f:.3f}, p = {p:.4f}')

---

## Summary of findings

| Finding | Result |
|---|---|
| Integration predicts engagement | r = .026, ns — **not supported** |
| Assimilation predicts engagement | r = .177, p = .019 — **small but significant** |
| Separation predicts stress | r = .211, p = .005 — **significant** |
| Same field → higher engagement | M = 4.34 vs 3.41, p < .001, d = .68 |
| Same field → lower stress | M = 1.58 vs 1.99, p = .008 |
| Field moderates stress-engagement | ΔR² = .000, p = .83 — **not supported** |
| Strongest predictor in regression | Same field (β = .338, p = .004) |

**Key conclusion:** Occupational continuity — working in the same professional field as in Venezuela — is the strongest predictor of work engagement in this sample, operating independently from acculturation stress.